<a href="https://colab.research.google.com/github/drchoi1/LING609_demo/blob/main/LING609_%60brms%60_logistic_regression_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LING609 Fall 2026: Logistic regression example

This Google Colab notebook accompanies in-class lecture on 10/22. It implements the logistic analysis from Nicenboim et al's Chapter 4.


In [ ]:
### This code chunk sets us up to run. It'll take about 20 minutes.
### Please go to "Runtime > Restart session" before running this chunk.
### install brms / cmdstanr / tidyverse
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
install.packages(c("brms", "emmeans", "dplyr", "ggplot2", "cmdstanr", "tidyverse", "bayesplot"),
                 repos = c("https://stan-dev.r-universe.dev",
                           getOption("repos")))
### load cmdstanr and set number of cores
library(cmdstanr)
install_cmdstan(cores = 2)

### load libraries that will be used in exercises
install.packages("remotes")
remotes::install_github("bnicenboim/bcogsci")
library(brms)
library(bcogsci)
library(emmeans)
library(bayesplot)
library(dplyr)
library(ggplot2)
options(brms.backend = "cmdstanr", mc.cores = 2)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘StanHeaders’, ‘inline’, ‘gridExtra’, ‘RcppParallel’, ‘QuickJSR’, ‘RcppEigen’, ‘BH’, ‘tensorA’, ‘distributional’, ‘Brobdingnag’, ‘globals’, ‘listenv’, ‘parallelly’, ‘quadprog’, ‘plyr’, ‘rstan’, ‘loo’, ‘posterior’, ‘rstantools’, ‘bridgesampling’, ‘future’, ‘future.apply’, ‘matrixStats’, ‘nleqslv’, ‘coda’, ‘abind’, ‘estimability’, ‘mvtnorm’, ‘numDeriv’, ‘checkmate’, ‘ggdist’, ‘ggridges’, ‘reshape2’


This is cmdstanr version 0.9.0

- CmdStanR documentation and vignettes: mc-stan.org/cmdstanr

- Use set_cmdstan_path() to set the path to CmdStan

- Use install_cmdstan() to install CmdStan

The C++ toolchain required for CmdStan is setup properly!

* Latest CmdStan release is v2.40.0

* Installing CmdStan v2.40.0 in /root/.cmdstan/cmdstan-2.40.0

* Downloading cmdstan-2.40.0.tar.gz from GitHub...

* Download complete

* Unpacking archive...

* Building CmdStan binaries...



g++ -Wno-deprecated-declarations -std=c++17 -pthread -D_REENTRANT -Wno-sign-compare -Wno-ignored-attributes -Wno-class-memaccess      -isystem stan/lib/stan_math/lib/tbb_2020.3/include    -O3 -I src -I stan/src -I stan/lib/rapidjson_1.1.0/ -I lib/CLI11-1.9.1/ -I stan/lib/stan_math/ -isystem stan/lib/stan_math/lib/eigen_5.0.1 -isystem stan/lib/stan_math/lib/boost_1.87.0 -isystem stan/lib/stan_math/lib/sundials_6.1.1/include -isystem stan/lib/stan_math/lib/sundials_6.1.1/src/sundials    -DBOOST_DISABLE_ASSERTS          -c -MT src/cmdstan/main.o -MM -E -MG -MP -MF src/cmdstan/main.d src/cmdstan/main.cpp
g++ -Wno-deprecated-declarations -std=c++17 -pthread -D_REENTRANT -Wno-sign-compare -Wno-ignored-attributes -Wno-class-memaccess      -isystem stan/lib/stan_math/lib/tbb_2020.3/include    -O3 -I src -I stan/src -I stan/lib/rapidjson_1.1.0/ -I lib/CLI11-1.9.1/ -I stan/lib/stan_math/ -isystem stan/lib/stan_math/lib/eigen_5.0.1 -isystem stan/lib/stan_math/lib/boost_1.87.0 -isystem stan/lib/st

* Finished installing CmdStan to /root/.cmdstan/cmdstan-2.40.0


CmdStan path set to: /root/.cmdstan/cmdstan-2.40.0

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)




rbibutils (NA -> 2.4.1  ) [CRAN]
rootSolve (NA -> 1.8.2.4) [CRAN]
Rdpack    (NA -> 2.6.6  ) [CRAN]


Installing 3 packages: rbibutils, rootSolve, Rdpack

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
### Basic fitting code chunk
data("df_recall")
head(df_recall)

## Center my predictor

## Summarize accuracy:
df_summary <- df_recall %>% group_by(c_set_size) %>% summarize(mean.acc = mean(correct), n = n())

## Basic logistic regression:



In [ ]:
### Summarizing and inspecting the model code chunk

## Summarize the whole model

## Plot the whole model

## Summarize the posterior over key variables only

## Extract the posterior samples themselves from the model themselves

## Recall that the intercept is the log-odds of a correct answer at the mean set size value. We can convert to different scales:

## First, convert log-odds to odds:

## Next, convert log-odds to probabilities:

## Let's convert to probabilities/accuracies and then take the average.

## How well does this align with the empirical data?

## Reporting odds for beta

## Statistical hypothesis: B < 0. We can directly inspect the samples to calculate this:


In [ ]:
## Let's look at accuracy rate by condition for our posterior predictive check
## Do we feel this is acceptable?
pp_check(fit_recall, type = "stat_grouped", stat = "mean", group = "c_set_size", ndraws = 1000)



In [ ]:
### Introduction to emmmeans
library(emm)
emm <- emmeans(fit_recall, ~ c_set_size, epred = TRUE, at = list(c_set_size = c(-3, -1, 1, 3)))
emm_summary <- as.data.frame(summary(emm, point.est = mean))

### Compare model estimated means, to observed means
emm_summary
df_summary

In [ ]:
#### Code for plotting
### First, combine predicted and observed means into a single dataframe
### then add a new 'cond' column that is a 'string' we can use for plotting.
plot_df <- left_join(df_summary, emm_summary, by = c("c_set_size")) %>%
  mutate(cond = as.character(c_set_size+5))

#### Plot empirical and model-fitted estimates
ggplot(plot_df, aes(x = cond)) +
  geom_col(aes(y = mean.acc), fill = "grey80", width = 0.6) +              ### Create columns; this references the 'mean.acc' variable
  geom_errorbar(aes(ymin = lower.HPD, ymax = upper.HPD), width = 0.15) +  ### Create error bars; this references the 'lower.HPD' and 'upper.HPD' variables
  geom_point(aes(y = emmean), size = 3) +                                  ### Create point; this references the 'emmean' variable
  coord_cartesian(ylim = c(0, 1), expand = FALSE) +                        ### Set y-limits to 0,1; standard for accuracy.
  labs(x = "Set size", y = "Accuracy") +
  theme_minimal()

### Suggested caption: "Bars represented empirical accuracy per condition. Points and error bars represent model estimate with 95% HPD interval."
